In [2]:
from manim import *
from scipy.integrate import solve_ivp

ModuleNotFoundError: No module named 'manim'

In [53]:
class GlowDot(Dot):
    def __init__(self, point=ORIGIN, color=WHITE, radius=0.1, glow_radius=0.3, **kwargs):
        super().__init__(point, color=color, radius=radius, **kwargs)
        
        # Create glow layers (multiple circles with decreasing opacity)
        self.glow = VGroup()
        for i in range(5):
            glow_circle = Dot(
                point=point,
                color=color,
                radius=radius + i * glow_radius / 5,
                fill_opacity=0.3 - i * 0.05
            )
            self.glow.add(glow_circle)
        
        # Add glow behind the main dot
        self.add_to_back(self.glow)
    
    def set_glow_color(self, color):
        self.glow.set_color(color)
        return self

In [ ]:
def lorenz_system(t, state, sigma=10, rho=10, beta=8/3):
  x, y, z = state
  dxdt = sigma * (y - x)
  dydt = x * (rho - z) - y
  dzdt = x * y - beta * z
  return [dxdt, dydt, dzdt]

def ode_solution_points(function, state0, time, dt=0.01):
  solution = solve_ivp(
    function,
    t_span=(0, time),
    y0=state0,
    t_eval=np.arange(0, time, dt)
  )
  return solution.y.T


class LorenzAttractor(ThreeDScene):
  def construct(self):
    FRAME_HEIGHT = config.frame_height
    FRAME_WIDTH = config.frame_width

    axes = ThreeDAxes(
        x_range=(-50, 50, 5),
        y_range=(-50, 50, 5),
        z_range=(0, 50, 5),
        x_length=16,
        y_length=16,
        z_length=8
    )
    
    # Deprecating soon!
    axes.set_width(FRAME_WIDTH)
    axes.center()

    self.add(axes)

    self.set_camera_orientation(
        phi=70 * DEGREES,    # Looking down from above
        theta=45 * DEGREES,  # Positioned to the right
    )

    template = TexTemplate()
    template.add_to_preamble(r"\usepackage{amsmath}")
    
    equations = MathTex(
      R"""
      \begin{aligned}
      \frac{dx}{dt} &= \sigma(y-x) \\
      \frac{dy}{dt} &= x(\rho-z)-y \\
      \frac{dz}{dt} &= xy-\beta z
      \end{aligned}
      """,
      tex_template=template,
      font_size=30
    )
    equations.set_color_by_tex("x", RED)
    equations.set_color_by_tex("y", GREEN)
    equations.set_color_by_tex("z", BLUE)

    bg = SurroundingRectangle(equations, color=GRAY, fill_opacity=0.2, buff=0.3, corner_radius=0.1)
    group = VGroup(equations, bg)
    group.to_corner(UL)

    equations.set_stroke(background=True, width=0.5)

    self.add_fixed_in_frame_mobjects(equations, bg)
    
    self.play(Write(equations), Create(bg))
    self.wait(3)

    self.move_camera(phi=70 * DEGREES, theta=45 * DEGREES, run_time=5)
    self.wait()
    

    # Compute set of solutions. 

    epsilon = 1e-5
    evolution_time = 30
    n_points = 10
    states = [
      [10, 10, 10 + n * epsilon]
      for n in range(n_points)
    ]
    colors = color_gradient([PURPLE, PINK], len(states))

    curves = VGroup()
    for state, color in zip(states, colors):
      points = ode_solution_points(lorenz_system, state, evolution_time)
      converted_points = np.array([axes.c2p(*points.T)])
      curve = VMobject().set_points_smoothly(converted_points)
      curve.set_stroke(color, width=1, opacity=0.25)
      curves.add(curve)

    curves.set_stroke(width=2, opacity=1)

    # Create bunch of glowing dots and unpack it into a Group.
    dots = Group(*(GlowDot(color=color, radius=0.25) for color in colors))

    def update_dots(dots, curves = curves):
      for dot, curve in zip(dots, curves):
        dot.move_to(curve.get_end())

    dots.add_updater(update_dots)

    tails = VGroup(
      *(TracedPath(dot.get_center, stroke_width=2, stroke_opacity=0.8).match_color(dot)
        for dot in dots)
    )
  
    self.add(dots, tails)
    curves.set_opacity(0)
    self.play(
      Create(curves, rate_func = linear),
      run_time = evolution_time
    )
    self.wait()

    self.move_camera(theta=360 * DEGREES, run_time=6)
    self.wait()





In [65]:
manim -pql LorenzAttractor

Manim Community v0.20.0

/tmp/ipykernel_97428/4060902901.py:33: DeprecationWarning: This method is not guaranteed to stay around. Please prefer setting the attribute normally or with Mobject.set().
  axes.set_width(FRAME_WIDTH)


[02/26/26 19:38:19] INFO     Animation 0 : Partial movie file written in                   ]8;id=329832;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=438759;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py#590\590]8;;\
                             '/home/dragi/Documents/manimations/code/media/videos/code/480                         
                             p15/partial_movie_files/LorenzAttractor/3590453459_2025994594                         
                             _3341102081.mp4'                                                                      

[02/26/26 19:38:20] INFO     Animation 1 : Partial movie file written in                   ]8;id=294643;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=727842;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py#590\590]8;;\
                             '/home/dragi/Documents/manimations/code/media/videos/code/480                         
                             p15/partial_movie_files/LorenzAttractor/1420018040_3879069619                         
                             _2029588275.mp4'                                                                      

[02/26/26 19:38:31] INFO     Animation 2 : Partial movie file written in                   ]8;id=771435;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=629885;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py#590\590]8;;\
                             '/home/dragi/Documents/manimations/code/media/videos/code/480                         
                             p15/partial_movie_files/LorenzAttractor/2221356904_2711168858                         
                             _753936057.mp4'                                                                       

...

[02/26/26 19:40:01] INFO     Animation 7 : Partial movie file written in                   ]8;id=614520;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=54189;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py#590\590]8;;\
                             '/home/dragi/Documents/manimations/code/media/videos/code/480                         
                             p15/partial_movie_files/LorenzAttractor/1561453711_2068277886                         
                             _577186045.mp4'                                                                       

                    INFO     Combining to Movie file.                                      ]8;id=195962;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=686068;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py#742\742]8;;\

                    INFO                                                                   ]8;id=653124;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=195734;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene_file_writer.py#893\893]8;;\
                             File ready at                                                                         
                             '/home/dragi/Documents/manimations/code/media/videos/code/480                         
                             p15/LorenzAttractor.mp4'                                                              
                                                                                                                   

                    INFO     Rendered LorenzAttractor                                                  ]8;id=101583;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=849130;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/scene/scene.py#278\278]8;;\
                             Played 8 animations                                                                   

[02/26/26 19:40:02] INFO     Previewed File at:                                                     ]8;id=963601;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/utils/file_ops.py\file_ops.py]8;;\:]8;id=760481;file:///home/dragi/Documents/manimations/.venv/lib/python3.14/site-packages/manim/utils/file_ops.py#236\236]8;;\
                             '/home/dragi/Documents/manimations/code/media/videos/code/480p15/Loren                
                             zAttractor.mp4'                                                                       